Plan: Instead of using top-10 approach and using absolute values, try clustering based on the results of power perturbation with signed values

perhaps also use time-point groups for clustering results


In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from scipy.cluster.hierarchy import fcluster
import itertools
from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:

def calculate_diff_per_channel(pred_label_original, freq_bands, amplification_factors, ch_names, subject_index=2, take_abs=False):
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        for factor in amplification_factors:
            perturbed_data = np.load(f"perturbed_predictions/perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}.npy", allow_pickle=True).item()
            mean_diff_per_channel[band_name][factor] = {}
            median_diff_per_channel[band_name][factor] = {}
            for ch_name in ch_names:
                perturbed_amplitude = perturbed_data[ch_name][0]
                if take_abs:
                    diff = np.abs(pred_label_original - perturbed_amplitude)
                else:
                    diff = pred_label_original - perturbed_amplitude
               
                mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
    
    return mean_diff_per_channel, median_diff_per_channel

In [ ]:
def get_top_channels(median_diff_per_channel, freq_bands, amplification_factors, top_k=10):
    top_channels_median = {}
    prediction_diff_median = {}
    
    for band_name in freq_bands.keys():
        top_channels_median[band_name] = {}
        prediction_diff_median[band_name] = {}
        
        for factor in amplification_factors:
            median_diffs = median_diff_per_channel[band_name][factor]
            
            # Sort the channels based on their differences
            sorted_median_diffs = sorted(median_diffs.items(), key=lambda item: item[1], reverse=True)
            
            # Select the top k channels
            top_channels_median[band_name][factor] = [ch for ch, _ in sorted_median_diffs[:top_k]]
            
            # Store the prediction differences for the top k channels
            prediction_diff_median[band_name][factor] = {ch: median_diffs[ch] for ch in top_channels_median[band_name][factor]}
    

    return top_channels_median,prediction_diff_median


In [ ]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=2, rep=1):
    data_dir = f"/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/gradshap_explanations_rep_{rep}"
    file_path = os.path.join(data_dir, f"gradshap_data_subject_{subject_index}_rep_{rep}.npy")

    subject_data = np.load(file_path, allow_pickle=True).item()
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [ ]:
def load_subject_topk(subject_index=2):
    topk = np.load("top_k_abs.npy", allow_pickle=True).item()
    return topk[subject_index]

In [ ]:
def get_top_k_keys(d, k):
    """
    Returns the top k keys in a dictionary that have the highest values.

    Parameters:
    d (dict): The input dictionary.
    k (int): The number of top keys to return.

    Returns:
    list: A list of the top k keys with the highest values.
    """
    # Sort the dictionary by values in descending order and get the top k keys
    top_k_keys = sorted(d, key=d.get, reverse=True)[:k]
    return top_k_keys

# Example usage
d = {'a': 10, 'b': 20, 'c': 15, 'd': 5, 'e': 25}
k = 3
print(get_top_k_keys(d, k))  # Output: ['e', 'b', 'c']

In [ ]:
freq_bands = {
              "delta": (0, 4),
              "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
amplification_factors = [0.5,0.8,0.9,1.1,1.2,1.5]

In [ ]:
dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/perturbed_predictions"

In [ ]:
cfg = load_config()
pairs = list(itertools.combinations(cfg.dataset.test_subject_indices, 2)) 

In [ ]:
def load_data_all_subjects():
    all_subjects_data = {}
    for subject_index in cfg.dataset.test_subject_indices:
        predictions, uncertainties, _, ch_names = load_predicted_amplitude_for_subject(subject_index)
        top_k = load_subject_topk(subject_index)
        

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info
        all_subjects_data[subject_index] = {"predictions": predictions, "uncertainties": uncertainties, "top_k": top_k, "ch_names": ch_names, "info_subj": info_subj}
    return all_subjects_data

In [ ]:
all_subjects_data = load_data_all_subjects()

In [ ]:
 #   mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(original_predictions, freq_bands, amplification_factors, ch_names, #subject_index=subject_index)
def median_difference_all_subjects_dw(data_all_subjects, take_abs=False, rep=1):
    distance_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_distance"
    pert_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_power"
    median_diff_per_channel_all_subjects = {}
    mean_diff_per_channel_all_subjects = {}
    std_diff_per_channel_all_subjects = {}
    for subject_index, data in data_all_subjects.items():
        predictions = data["predictions"]
        ch_names = data["ch_names"]
        median_diff_per_channel = {}
        mean_diff_per_channel = {}
        std_diff_per_channel = {}
        for band_name, (low_freq, high_freq) in freq_bands.items():
            median_diff_per_channel[band_name] = {}
            mean_diff_per_channel[band_name] = {}
            std_diff_per_channel[band_name] = {}
            for factor in amplification_factors:
            
                file_path_distance = f"parallel_distance_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
                load_path_distance = os.path.join(distance_dir, file_path_distance)
                distances = np.load(load_path_distance, allow_pickle=True).item()

                perturbed_data = np.load(f"{pert_dir}/parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy", allow_pickle=True).item()

                median_diff_per_channel[band_name][factor] = {}
                mean_diff_per_channel[band_name][factor] = {}
                std_diff_per_channel[band_name][factor] = {}
                for ch_name in ch_names:
                    perturbed_amplitude = perturbed_data[ch_name]
                    if take_abs:
                        diff = np.abs(predictions - perturbed_amplitude)/distances[ch_name]         
                    else:
                        diff = (predictions - perturbed_amplitude)/distances[ch_name]

                    median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
                    mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                    std_diff_per_channel[band_name][factor][ch_name] = np.std(diff)
        median_diff_per_channel_all_subjects[subject_index] = median_diff_per_channel
        mean_diff_per_channel_all_subjects[subject_index] = mean_diff_per_channel
        std_diff_per_channel_all_subjects[subject_index] = std_diff_per_channel
    return median_diff_per_channel_all_subjects


In [ ]:
median_diff_per_channel_all_subjects = median_difference_all_subjects_dw(all_subjects_data)

In [ ]:
def get_top_channels_all_subjects(median_diff_per_channel_all_subjects):
    top_channels_median_all_subjects = {}
    prediction_diff_median_all_subjects = {}
    for subject_index, median_diff_per_channel in median_diff_per_channel_all_subjects.items():
        top_channels_median,prediction_diff_median = get_top_channels(median_diff_per_channel, freq_bands, amplification_factors)
        top_channels_median_all_subjects[subject_index] = top_channels_median
        prediction_diff_median_all_subjects[subject_index] = prediction_diff_median
    return top_channels_median_all_subjects, prediction_diff_median_all_subjects

In [ ]:
top_channels_median_all_subjects, prediction_diff_median_all_subjects = get_top_channels_all_subjects(median_diff_per_channel_all_subjects)

# similarity/distance matrices

# clustering

In [ ]:
def get_common_channels(all_subjects_data):
    # Get the first subject's channel names as a set
    first_subject = list(all_subjects_data.keys())[0]
    common_channels = set(all_subjects_data[first_subject]["ch_names"])
    
    # Intersect with each other subject's channels
    for subject_id in all_subjects_data.keys():
        subject_channels = set(all_subjects_data[subject_id]["ch_names"])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(all_subjects_data)} subjects")
    print("Common channels:", common_channels)
    
    return common_channels

In [ ]:
common_channels = get_common_channels(all_subjects_data)

analyze which channel names are in all subject datas.
remove all other channels from the subject data for all subjects

need to find out which indices they correspond to in different subjects
only compute results based these channels?
Potentially good enough to remove all channels from the top 10 that are not in the common channels

In [ ]:
def get_cluster_labels(linkage_matrix, n_clusters=4):
    """Extract cluster labels from linkage matrix"""

    return fcluster(linkage_matrix, n_clusters, criterion='maxclust')

# across amplification factors

while positions seem correct, the names of the positions are wrong right now

# cluster based on rank correlations

In [ ]:
def get_common_channels(all_subjects_data):
    # Get the first subject's channel names as a set
    first_subject = list(all_subjects_data.keys())[0]
    common_channels = set(all_subjects_data[first_subject]["ch_names"])
    
    # Intersect with each other subject's channels
    for subject_id in all_subjects_data.keys():
        subject_channels = set(all_subjects_data[subject_id]["ch_names"])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(all_subjects_data)} subjects")
    print("Common channels:", common_channels)
    
    return common_channels
common_channels = get_common_channels(all_subjects_data)

In [ ]:
import scipy
def calculate_pairwise_correlations(median_diff1, median_diff2, freq_bands, amplification_factors, common_channels):

    correlation_results = {}
    correlation_results_mean = {}

    for band_name in freq_bands.keys():
        correlation_results[band_name] = {}
        for factor in amplification_factors:
            # Extract the top 10 channels for both subjects
            data1 =median_diff1[band_name][factor]
            data2 = median_diff2[band_name][factor]
            # Extract values for common channels only
            common_channel_values1 = np.array([data1[ch] for ch in common_channels])
            common_channel_values2 = np.array([data2[ch] for ch in common_channels])
            
            # Calculate correlations
            pearson_corr = np.corrcoef(common_channel_values1, common_channel_values2)[0,1]
            spearman_corr = scipy.stats.spearmanr(common_channel_values1, common_channel_values2).correlation
            
            # Store results
            correlation_results[band_name][factor] = {
                'pearson': pearson_corr,
                'spearman': spearman_corr
            }
            # Calculate agreement
        # compute the mean correlation across factors
        correlation_results_mean[band_name] = {
            'pearson': np.mean([correlation_results[band_name][factor]['pearson'] for factor in amplification_factors]),
            'spearman': np.mean([correlation_results[band_name][factor]['spearman'] for factor in amplification_factors])
        }

    return correlation_results, correlation_results_mean

In [ ]:
def plot_agreement_matrix_correlations(correlation_results, sub1_index, sub2_index):
    # plot a matrix with separate row for each frequency band and separate column for each amplification factor
    bands = list(correlation_results.keys())
    factors = list(correlation_results[bands[0]].keys())
    correlation_matrix_pearson = np.zeros((len(bands), len(factors)))
    correlation_matrix_spearman = np.zeros((len(bands), len(factors)))
    
    fig,axs = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)
    fig.subplots_adjust(wspace=-0.2)  # Reduce horizontal space between subplots
    fig.suptitle(f'Correlation between subjects {sub1_index} and {sub2_index}')
    
    # Fill correlation matrices
    for i, band in enumerate(bands):
        for j, factor in enumerate(factors):
            correlation_matrix_pearson[i, j] = correlation_results[band][factor]['pearson']
            correlation_matrix_spearman[i, j] = correlation_results[band][factor]['spearman']

    
    im_pearson = axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=0, vmax=1)
    im_spearman = axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=0, vmax=1)

    # Add single colorbar for both plots
    cbar = fig.colorbar(im_spearman, ax=axs.ravel().tolist(), location='bottom', shrink=0.35)
    cbar.set_label('Correlation')

    # Set axis labels
    for ax in axs:
        ax.set_xticks(np.arange(len(factors)))
        ax.set_yticks(np.arange(len(bands)))
        ax.set_xticklabels(factors)
        ax.set_yticklabels(bands)
        ax.set_xlabel('Amplification factors')
        ax.set_ylabel('Frequency Bands')
        plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
    
    axs[0].set_title('Pearson correlation')
    axs[1].set_title('Spearman correlation')

    # Add text annotations
    for i in range(len(bands)):
        for j in range(len(factors)):
            axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                       ha="center", va="center", color="white")
            axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                       ha="center", va="center", color="white")


In [ ]:
cfg = load_config()
rank_correlations_mean_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    correlation_results, correlation_results_mean = calculate_pairwise_correlations(median_diff_per_channel_all_subjects[subject_index1], median_diff_per_channel_all_subjects[subject_index2], freq_bands, amplification_factors, common_channels)
    rank_correlations_mean_all_pairs[(subject_index1, subject_index2)] = correlation_results_mean
    #plot_agreement_matrix_correlations(correlation_results, subject_index1, subject_index2)
    

In [ ]:
correlation_results

In [ ]:
def plot_agreement_matrix_correlations_all_subjects(mean_correlations_all_subject_pairs, freq_band="alpha", take_abs=False, plot=True):
    # for a given frequency band and amplification factor, plot the correlation between all subjects
    # subjcet indicies are on rows and columns
    # the value in each cell is the mean correlation across all factors
    #
    correlation_matrix_pearson = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    correlation_matrix_spearman = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    subject_indices = list(cfg.dataset.test_subject_indices)                             
    for subj1,subj2 in itertools.combinations(subject_indices,2):
        if take_abs:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
        else:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']
    
        # use the correlations matrices as distance metrics:
    distance_matrix_pearson = 1 - correlation_matrix_pearson
    distance_matrix_spearman = 1 - correlation_matrix_spearman
    os.makedirs("distance_matrices", exist_ok=True)
    np.save(f"distance_matrices/power_correlation_matrix_pearson_freq_band_{freq_band}.npy", distance_matrix_pearson)
    np.save(f"distance_matrices/power_correlation_matrix_spearman_freq_band_{freq_band}.npy", distance_matrix_spearman)

    if plot:
        fig,axs = plt.subplots(nrows=2, ncols=1, figsize=(25, 25))

        axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=-1, vmax=1)
        axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=-1, vmax=1)

        # Set axis labels
        for ax in axs:
            ax.set_xticks(np.arange(len(subject_indices)))
            ax.set_yticks(np.arange(len(subject_indices)))
            ax.set_xticklabels(subject_indices)
            ax.set_yticklabels(subject_indices)
            ax.set_xlabel('Subject indices')
            ax.set_ylabel('Subject indices')
            plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
        
        axs[0].set_title('Pearson correlation')
        axs[1].set_title('Spearman correlation')

        # Add text annotations
        for i in range(len(subject_indices)):
            for j in range(len(subject_indices)):
                axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                           ha="center", va="center", color="white")
                axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                           ha="center", va="center", color="white")
        
        return fig, correlation_matrix_pearson, correlation_matrix_spearman
    
    return None, correlation_matrix_pearson, correlation_matrix_spearman


                                           

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="theta", take_abs=False)

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="delta", take_abs=False)

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="alpha", take_abs=False)

In [ ]:

plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="beta", take_abs=False)

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="gamma", take_abs=False)

## dont take absolute value in median differences

In [ ]:
def plot_agreement_matrix_correlations_all_subjects(mean_correlations_all_subject_pairs, freq_band="alpha", take_abs=False, plot=True):
    # for a given frequency band and amplification factor, plot the correlation between all subjects
    # subjcet indicies are on rows and columns
    # the value in each cell is the mean correlation across all factors
    #
    correlation_matrix_pearson = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    correlation_matrix_spearman = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    subject_indices = list(cfg.dataset.test_subject_indices)                             
    for subj1,subj2 in itertools.combinations(subject_indices,2):
        if take_abs:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
        else:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']

        # use the correlations matrices as distance metrics:
    distance_matrix_pearson = 1 - correlation_matrix_pearson
    distance_matrix_spearman = 1 - correlation_matrix_spearman
    os.makedirs("distance_matrices", exist_ok=True)
    np.save(f"distance_matrices/power_correlation_matrix_pearson_freq_band_{freq_band}.npy", distance_matrix_pearson)
    np.save(f"distance_matrices/power_correlation_matrix_spearman_freq_band_{freq_band}.npy", distance_matrix_spearman)

    if plot:
        fig,axs = plt.subplots(nrows=2, ncols=1, figsize=(25, 25))

        axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=-1, vmax=1)
        axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=-1, vmax=1)

        # Set axis labels
        for ax in axs:
            ax.set_xticks(np.arange(len(subject_indices)))
            ax.set_yticks(np.arange(len(subject_indices)))
            ax.set_xticklabels(subject_indices)
            ax.set_yticklabels(subject_indices)
            ax.set_xlabel('Subject indices')
            ax.set_ylabel('Subject indices')
            plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
        
        axs[0].set_title('Pearson correlation')
        axs[1].set_title('Spearman correlation')

        # Add text annotations
        for i in range(len(subject_indices)):
            for j in range(len(subject_indices)):
                axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                           ha="center", va="center", color="white")
                axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                           ha="center", va="center", color="white")
        
        return fig, correlation_matrix_pearson, correlation_matrix_spearman
    
    return None, correlation_matrix_pearson, correlation_matrix_spearman


                                           

In [ ]:
median_diff_per_channel_all_subjects = median_difference_all_subjects_dw(all_subjects_data, take_abs=False)
cfg = load_config()
rank_correlations_mean_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    correlation_results, correlation_results_mean = calculate_pairwise_correlations(median_diff_per_channel_all_subjects[subject_index1], median_diff_per_channel_all_subjects[subject_index2], freq_bands, amplification_factors, common_channels)
    rank_correlations_mean_all_pairs[(subject_index1, subject_index2)] = correlation_results_mean
    #plot_agreement_matrix_correlations(correlation_results, subject_index1, subject_index2)
    

In [ ]:
rank_correlations_mean_all_pairs

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import os

def cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="alpha", correlation_type='spearman', take_abs=False, n_clusters=4, save_path=None, ax=None, treshold=None):
    """
    Create a distance matrix from correlations, perform hierarchical clustering and visualize results.
    
    Parameters:
    rank_correlations_mean_all_pairs (dict): Dictionary of correlation values between subject pairs
    freq_band (str): Frequency band to use ('theta', 'delta', 'alpha', 'beta', 'gamma')
    correlation_type (str): Type of correlation to use ('pearson' or 'spearman')
    take_abs (bool): Whether to take absolute value of correlations before converting to distance
    n_clusters (int): Number of clusters to identify
    save_path (str, optional): Path to save the results. If None, results are not saved.
    ax (matplotlib.axes.Axes, optional): Axis to draw the dendrogram on. If None, a new figure is created.
    
    Returns:
    fig: Figure containing the dendrogram (if ax is None)
    cluster_labels: Cluster assignments for each subject
    distance_matrix: The computed distance matrix
    subject_ids: List of subject IDs used in clustering
    """
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    
    # Get the list of unique subject IDs
    subject_ids = set()
    for pair in rank_correlations_mean_all_pairs:
        subject_ids.add(pair[0])
        subject_ids.add(pair[1])
    subject_ids = sorted(list(subject_ids))
    n_subjects = len(subject_ids)
    
    # Create a mapping from subject ID to index
    subject_to_idx = {subj: idx for idx, subj in enumerate(subject_ids)}
    
    # Initialize the distance matrix with ones (maximum distance)
    distance_matrix = np.ones((n_subjects, n_subjects))
    np.fill_diagonal(distance_matrix, 0)  # Zero distance to self
    
    # Fill the distance matrix
    for pair, correlations in rank_correlations_mean_all_pairs.items():
        i, j = subject_to_idx[pair[0]], subject_to_idx[pair[1]]
        corr_value = correlations[freq_band][correlation_type]
        
        if take_abs:
            distance = 1 - abs(corr_value)
        else:
            distance = 1 - corr_value
            
        distance_matrix[i, j] = distance
        distance_matrix[j, i] = distance  # Matrix is symmetric
    
    # Convert the distance matrix to a condensed form for linkage
    np.save(f"distance_matrices/power_correlation_matrix_{correlation_type}_freq_band_{freq_band}.npy", distance_matrix)
    condensed_matrix = squareform(distance_matrix)
    
    # Compute the linkage matrix
    linkage_matrix = linkage(condensed_matrix, method='ward')
    
    # Get cluster assignments
    cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    
    # Create figure and axis if ax is not provided
    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 4))
    
    # Create color map for clusters
    color_palette = cm.rainbow(np.linspace(0, 1, n_clusters))
    cluster_colors = {i+1: color_palette[i] for i in range(n_clusters)}
    
    cluster_color_map = {}
    for i, label in enumerate(cluster_labels):
        cluster_color_map[str(subject_ids[i])] = cluster_colors[label]
    
    # Plot dendrogram
    dendrogram(
        linkage_matrix,
        labels=subject_ids,
        ax=ax,
        leaf_rotation=90,
        leaf_font_size=10,
        color_threshold=treshold
    )
    
    # Color the leaves based on cluster
    #for i, label in zip(range(n_subjects), ax.get_xticklabels()):
    #    label.set_color(cluster_color_map[label.get_text()])
    
    #ax.set_title(f'Hierarchical Clustering ({n_clusters} clusters)\n{correlation_type.capitalize()} Correlation - {freq_band.capitalize()} Band')
    ax.set_xlabel('Subject ID', fontsize=18)
    ax.set_ylabel('Distance', fontsize=18)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, fontsize=16)
    ax.set_yticklabels(ax.get_yticks(), fontsize=16)
    
    # Save results if path is provided
    if save_path and fig is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(f"{save_path}_dendrogram.png", dpi=300, bbox_inches='tight')
        np.save(f"{save_path}_distance_matrix.npy", distance_matrix)
    
    fig.savefig(f"hierarchical_power_spearman_{freq_band}.png", dpi=300)

    return fig, cluster_labels, distance_matrix, subject_ids


In [ ]:
def plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="alpha", normalize=False):         #   general idea for the subjects in a cluster, aggregate their values of median_differences and plot the topomap
    n_clusters = len(np.unique(cluster_labels))
    n_subjects = len(subject_ids)
    ch_names = all_subjects_data[1]["ch_names"]
    fig,axs = plt.subplots(nrows=1, ncols=n_clusters, figsize=(16, 4))
    #fig.suptitle('Topomap of Median Differences for Each Cluster')
    info = all_subjects_data[1]["info_subj"]
    for cluster_id in range(1, n_clusters + 1):
   
  
        # Find subjects in this cluster
        cluster_subjects = [subj for subj, label in zip(subject_ids, cluster_labels) if label == cluster_id]
        importance_channels_cluster = np.zeros(len(ch_names))
        for subject in cluster_subjects:
            subject_sum = 0
            median_diff_per_channel = median_diff_per_channel_all_subjects[subject]
            median_diff = median_diff_per_channel[band_name][1.5]
            if normalize:
                for ch in common_channels:
                    subject_sum += np.abs(median_diff[ch])

            for ch in common_channels:
                if normalize:
                    importance_channels_cluster[ch_names.index(ch)] += median_diff[ch] / subject_sum
                else:
                    importance_channels_cluster[ch_names.index(ch)] += median_diff[ch]
            # think about dividing results by number of subjects in the cluster 
     
     
        
        
        if n_clusters >1:
            axs[cluster_id-1].set_title(f'Cluster {cluster_id}, N: {len(cluster_subjects)}', fontsize=18)
            mne.viz.plot_topomap(importance_channels_cluster, info, axes=axs[cluster_id-1], show=False, names=ch_names)
        else:
            axs.set_title(f'Cluster {cluster_id}', fontsize=18)
            mne.viz.plot_topomap(importance_channels_cluster, info, axes=axs, show=False, names=ch_names)
    
    fig.savefig(f"topomap_cluster_power_spearman_{band_name}.png", dpi=300, bbox_inches='tight')


    

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster, cophenet
from sklearn.metrics import silhouette_score

def evaluate_n_clusters(distance_matrix, method='average', max_k=10):

# Store results
    results = {}


    Z = linkage(distance_matrix, method=method)
    silhouette_scores = []

    within_cluster_dists = []
    dunn_indices = []
    
    for k in range(2, max_k + 1):
        labels = fcluster(Z, k, criterion='maxclust')
        silhouette = silhouette_score(distance_matrix, labels, metric='precomputed')

        
        # Compute within-cluster sum of distances
        within_dist = 0
        for cluster in np.unique(labels):
            indices = np.where(labels == cluster)[0]
            if len(indices) > 1:
                within_dist += np.sum(distance_matrix[np.ix_(indices, indices)])
        within_cluster_dists.append(within_dist)
        
        # Calculate Dunn Index
        max_intra_cluster_dist = 0
        min_inter_cluster_dist = float('inf')
        
        # Find max intra-cluster distance
        for cluster in np.unique(labels):
            indices = np.where(labels == cluster)[0]
            if len(indices) > 1:
                cluster_dists = distance_matrix[np.ix_(indices, indices)]
                max_dist = np.max(cluster_dists)
                max_intra_cluster_dist = max(max_intra_cluster_dist, max_dist)
        
        # Find min inter-cluster distance
        for i, cluster1 in enumerate(np.unique(labels)):
            indices1 = np.where(labels == cluster1)[0]
            for cluster2 in np.unique(labels)[i+1:]:
                indices2 = np.where(labels == cluster2)[0]
                inter_dists = distance_matrix[np.ix_(indices1, indices2)]
                min_dist = np.min(inter_dists)
                min_inter_cluster_dist = min(min_inter_cluster_dist, min_dist)
        
        # Calculate Dunn Index (avoid division by zero)
        if max_intra_cluster_dist > 0:
            dunn_idx = min_inter_cluster_dist / max_intra_cluster_dist
        else:
            dunn_idx = float('inf')
        
        dunn_indices.append(dunn_idx)
        silhouette_scores.append(silhouette)

        

    
    results[method] = {
        'silhouette': silhouette_scores,
        'within_cluster_dist': within_cluster_dists,
        'dunn_index': dunn_indices
    }

    return results


In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="delta", correlation_type='spearman', take_abs=False, n_clusters=3, save_path=None, treshold=1.75)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

In [ ]:
_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="theta", correlation_type='spearman', take_abs=False, n_clusters=4, save_path=None, treshold=1.4)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="theta")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="theta", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

this one is either 2,3 or 5 clusters :(


In [ ]:
_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="alpha", correlation_type='spearman', take_abs=False, n_clusters=4, save_path=None, treshold=1.5)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="alpha")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="alpha", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

choose 3 or 4 for this

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="beta", correlation_type='spearman', take_abs=False, n_clusters=3, save_path=None, treshold=1.6)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="beta", normalize=False)
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="beta", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

this one is tough. its either 2 (hierarchical and shilouette), 4, or 6 (dunn, elbow)

In [ ]:
_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="gamma", correlation_type='spearman', take_abs=False, n_clusters=3, save_path=None, treshold=1.5)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="gamma")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="gamma", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

2 according to shilouette, 3 in hierachical 5 according to dunn 4 according to elbpw

# with pearson correlation

# get distance matrices

In [ ]:
def compute_distance_matrices(all_rank_correlations_all_pairs, factors, is_abs=False, band_name="alpha", correlation_type='spearman'):
    cfg = load_config()
    n_subjects = len(cfg.dataset.test_subject_indices)
    subjects_indices = np.arange(n_subjects) 
    subject_ids = cfg.dataset.test_subject_indices
    distance_matrices = {}
    for factor in factors:
        distance_matrix = np.zeros((n_subjects, n_subjects))
        for subject_index1, subject_index2 in itertools.combinations(subjects_indices ,2):
            #print(subject_ids[int(subject_index1)])
            correlation_results = all_rank_correlations_all_pairs[(subject_ids[int(subject_index1)], subject_ids[int(subject_index2)])]
            correlation_value = correlation_results[band_name][factor][correlation_type]
           
            distance = 1 - correlation_value

            distance_matrix[subject_index1, subject_index2] = distance
            distance_matrix[subject_index2, subject_index1] = distance
        distance_matrices[factor] = distance_matrix
    
    if is_abs:
        np.save(f"distance_matrices/power_correlation_matrix_{correlation_type}_{band_name}_abs.npy", distance_matrices)
    else:
        np.save(f"distance_matrices/power_correlation_matrix_{correlation_type}_{band_name}.npy", distance_matrices)
   





## abs

In [ ]:
median_diff_per_channel_all_subjects_abs = median_difference_all_subjects_dw(all_subjects_data, take_abs=True)

In [ ]:
cfg = load_config()
rank_correlations_mean_all_pairs_abs = {}
all_rank_correlations_all_pairs_abs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    correlation_results, correlation_results_mean = calculate_pairwise_correlations(median_diff_per_channel_all_subjects_abs[subject_index1], median_diff_per_channel_all_subjects_abs[subject_index2], freq_bands, amplification_factors, common_channels)
    rank_correlations_mean_all_pairs_abs[(subject_index1, subject_index2)] = correlation_results_mean
    all_rank_correlations_all_pairs_abs[(subject_index1, subject_index2)] = correlation_results
    #plot_agreement_matrix_correlations(correlation_results, subject_index1, subject_index2)

In [ ]:
for freq_band in freq_bands.keys():
    compute_distance_matrices(all_rank_correlations_all_pairs_abs, amplification_factors, is_abs=True, band_name=freq_band, correlation_type='spearman')

In [ ]:
for freq_band in freq_bands.keys():
    compute_distance_matrices(all_rank_correlations_all_pairs_abs, amplification_factors, is_abs=True, band_name=freq_band, correlation_type='pearson')

## non abs

In [ ]:
median_diff_per_channel_all_subjects = median_difference_all_subjects_dw(all_subjects_data, take_abs=False)

In [ ]:
cfg = load_config()
rank_correlations_mean_all_pairs = {}
all_rank_correlations_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    correlation_results, correlation_results_mean = calculate_pairwise_correlations(median_diff_per_channel_all_subjects[subject_index1], median_diff_per_channel_all_subjects[subject_index2], freq_bands, amplification_factors, common_channels)
    rank_correlations_mean_all_pairs[(subject_index1, subject_index2)] = correlation_results_mean
    all_rank_correlations_all_pairs[(subject_index1, subject_index2)] = correlation_results

In [ ]:
for freq_band in freq_bands.keys():
    compute_distance_matrices(all_rank_correlations_all_pairs, amplification_factors, is_abs=False, band_name=freq_band, correlation_type='spearman')

In [ ]:
for freq_band in freq_bands.keys():
    compute_distance_matrices(all_rank_correlations_all_pairs, amplification_factors, is_abs=False, band_name=freq_band, correlation_type='pearson')

# for abs values

In [ ]:
median_diff_per_channel_all_subjects = median_difference_all_subjects_dw(all_subjects_data, take_abs=True)
cfg = load_config()
rank_correlations_mean_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    correlation_results, correlation_results_mean = calculate_pairwise_correlations(median_diff_per_channel_all_subjects[subject_index1], median_diff_per_channel_all_subjects[subject_index2], freq_bands, amplification_factors, common_channels)
    rank_correlations_mean_all_pairs[(subject_index1, subject_index2)] = correlation_results_mean
    #plot_agreement_matrix_correlations(correlation_results, subject_index1, subject_index2)
    

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="delta", correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None, treshold=1.75)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="theta", correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None, treshold=1.75)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="theta")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="alpha", correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None, treshold=1.75)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="alpha")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="beta", correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None, treshold=1.75)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="beta")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_mean_all_pairs, freq_band="gamma", correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None, treshold=1.75)
plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="gamma")
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)